# Lab 01-09 · Multivariable Calculus I

Companion to chapter [01-09](../../chapters/01-09-multivariable-calculus-1.md). Handout: [README.md](README.md).

Run the cells in order. Cells marked **🔮 Predict** ask you to write a prediction *before* running.

In [1]:
import math, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass

## Part 1 — Partial derivatives, symbolically and numerically

In [2]:
VGS, VDS, k, VTH, VA = sp.symbols("V_GS V_DS k V_TH V_A", positive=True)
ID = sp.Rational(1, 2) * k * (VGS - VTH)**2 * (1 + VDS / VA)
gm, go = sp.diff(ID, VGS), sp.diff(ID, VDS)
print("g_m =", sp.simplify(gm)); print("g_o =", sp.simplify(go))
vals = {k: 2e-3, VTH: 0.5, VA: 10, VGS: 0.8, VDS: 1.0}
print(f"I_D = {float(ID.subs(vals))*1e6:.2f} uA, g_m = {float(gm.subs(vals))*1e3:.4f} mS, g_o = {float(go.subs(vals))*1e6:.3f} uS")

# central-difference check (01-03 sec. 3.9)
f = sp.lambdify((VGS, VDS), ID.subs({k: 2e-3, VTH: 0.5, VA: 10}))
h = 1e-5
print(f"numerical g_m = {(f(0.8+h,1.0)-f(0.8-h,1.0))/(2*h)*1e3:.4f} mS, g_o = {(f(0.8,1.0+h)-f(0.8,1.0-h))/(2*h)*1e6:.3f} uS")

g_m = k*(V_A + V_DS)*(V_GS - V_TH)/V_A
g_o = k*(V_GS - V_TH)**2/(2*V_A)
I_D = 99.00 uA, g_m = 0.6600 mS, g_o = 9.000 uS
numerical g_m = 0.6600 mS, g_o = 9.000 uS


## Part 2 — Gradient descent and conditioning

**🔮 Predict** the largest stable step size for $f = x^2 + 10y^2$, and how many steps from (1, 1) it takes to reach $f < 10^{-6}$ with the best fixed step.

In [3]:
def descend(cx, cy, eta, start=(1.0, 1.0), max_steps=10_000, tol=1e-6):
    x, y = start
    for n in range(max_steps):
        if cx * x * x + cy * y * y < tol:
            return n
        x, y = x - eta * 2 * cx * x, y - eta * 2 * cy * y
        if abs(x) > 1e6 or abs(y) > 1e6:
            return None
    return max_steps

for cy in [1, 10, 100]:
    limit = 2 / (2 * cy)                       # 2 / largest curvature
    best = min((descend(1, cy, e), e) for e in np.linspace(0.01 * limit, 0.999 * limit, 200)
               if descend(1, cy, e) is not None)
    print(f"f = x^2 + {cy:>3} y^2: stability limit eta < {limit:.4f}, best steps {best[0]:>5} at eta = {best[1]:.4f}")

f = x^2 +   1 y^2: stability limit eta < 1.0000, best steps     2 at eta = 0.4871
f = x^2 +  10 y^2: stability limit eta < 0.1000, best steps    36 at eta = 0.0875


f = x^2 + 100 y^2: stability limit eta < 0.0100, best steps   348 at eta = 0.0098


The number of steps grows roughly in proportion to the condition number, the ratio of the largest to the smallest curvature.

## Part 3 — Constrained optimization: Lagrange against a numerical optimizer

In [4]:
ks = np.array([0.1, 0.4, 0.9]); F = 3.4
res = minimize(lambda f: np.sum(ks * f**3), x0=np.full(3, F / 3), method="SLSQP",
               constraints=[{"type": "eq", "fun": lambda f: f.sum() - F}], bounds=[(0, None)] * 3)
w = 1 / np.sqrt(ks); lag = F * w / w.sum()
print("numerical optimum:", np.round(res.x, 4), f"P = {res.fun:.4f} W")
print("Lagrange closed form:", np.round(lag, 4), f"P = {np.sum(ks * lag**3):.4f} W")

def waterfill(N, P):
    N = np.asarray(N, float)
    for m in range(len(N), 0, -1):            # try using the m best channels
        idx = np.argsort(N)[:m]
        mu = (P + N[idx].sum()) / m
        if mu > N[idx].max():
            p = np.zeros_like(N); p[idx] = mu - N[idx]
            return p, mu
for P in [6, 1]:
    p, mu = waterfill([1, 3], P)
    C = np.sum(np.log2(1 + p / np.array([1, 3])))
    print(f"P = {P} mW: allocation {p}, water level {mu:.2f}, capacity {C:.4f} b/s/Hz")

numerical optimum: [1.8544 0.9271 0.6184] P = 1.1694 W
Lagrange closed form: [1.8545 0.9273 0.6182] P = 1.1694 W


P = 6 mW: allocation [4. 2.], water level 5.00, capacity 3.0589 b/s/Hz
P = 1 mW: allocation [1. 0.], water level 2.00, capacity 1.0000 b/s/Hz


## Part 4 — ngspice reads partial derivatives from device models

In [5]:
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "mosfet_tempco.cir"], capture_output=True, text=True).stdout
    for l in out.splitlines():
        if "=" in l and l.split("=")[0].strip() in ("id", "gm", "gds", "va", "dvdt"):
            print("   " + l.strip())
else:
    print("ngspice not found; see SETUP.md. Skipping.")

   id = 9.9E-05
   gm = 0.00066
   gds = 9E-06
   va = 0.654791
   dvdt = -0.00180914


## Your turn

1. In Part 2, add momentum ($v \leftarrow \beta v - \eta\nabla f$, $x \leftarrow x + v$ with β = 0.9) and compare step counts for the cy = 100 case.
2. In Part 3, give each core a fixed idle power of 0.3 W when active. Is it still best to use all three cores at F = 3.4 GHz?